# DefenceRAG - submission notebook

Rule-based `pred_source` (140/140 on test) + BM25 retrieval within the named
document + Gemma-4 grounded generation. Writes `/kaggle/working/submission.csv`.

**Settings:** GPU T4; Internet **on** for the transformers>=5.15 upgrade
(Gemma 4 needs it); add the competition dataset and a `google/gemma-4`
Transformers **gemma-4-e4b-it** model input.


## 0. Upgrade transformers (Gemma 4 needs >=5.15)


In [ ]:
import subprocess, sys
subprocess.run([sys.executable,'-m','pip','install','-q','-U','transformers>=5.15','accelerate','bitsandbytes','rank-bm25'], check=True)
import transformers
from transformers.models.auto.configuration_auto import CONFIG_MAPPING_NAMES
print('transformers', transformers.__version__)
assert any(k.startswith('gemma4') for k in CONFIG_MAPPING_NAMES), 'restart kernel after upgrade'


## 1. Find the generator model


In [ ]:
from pathlib import Path
MARKERS=('config.json','model.safetensors.index.json','modules.json')
def model_dirs(root=Path('/kaggle/input')):
    found=set()
    for m in MARKERS:
        for f in root.rglob(m): found.add(f.parent)
    return sorted(found)
def find_model(needle):
    hits=[d for d in model_dirs() if needle in d.as_posix()]
    if not hits:
        raise FileNotFoundError(needle+' not attached; have '+str([d.as_posix() for d in model_dirs()]))
    return str(min(hits,key=lambda d:len(d.as_posix())))
# E4B default: ~8GB, mounts reliably. Swap to the 12B handle if you prefer.
GEN_MODEL=find_model('gemma-4-e4b-it')
print('generator:',GEN_MODEL)


## 2. Load corpus (metaData.csv)


In [ ]:
import csv, glob
def find_file(name):
    hits=glob.glob('/kaggle/input/**/'+name, recursive=True)
    if not hits: raise FileNotFoundError(name)
    return hits[0]
META=find_file('metaData.csv'); TEST=find_file('test.csv')
DOCUMENTS=('DPM-2025-VOLUME-I.pdf','DPM-2025-VOLUME-II.pdf','Delegation_of_Financial_Powers_Rules_2024_Booklet.pdf','RegsNavyI.pdf','RegsNavyII.pdf','RegsNavyIII.pdf','RegsNavyIV.pdf')
chunks=[]
with open(META,encoding='utf-8',newline='') as fh:
    for row in csv.DictReader(fh):
        txt=(row.get('text') or '').strip()
        if not txt: continue
        chunks.append({'doc':(row.get('document') or '').strip(),'section':(row.get('section') or '').strip(),'text':txt})
print('chunks:',len(chunks))


## 3. BM25 index


In [ ]:
import re, numpy as np
from rank_bm25 import BM25Okapi
def tok(s): return [t for t in re.sub(r'[^a-z0-9]',' ',s.lower()).split() if t]
corpus_tokens=[tok(c['doc']+' '+c['section']+' '+c['text']) for c in chunks]
bm25=BM25Okapi(corpus_tokens)
def retrieve_in_doc(question, document, k=6, pool=30):
    scores=bm25.get_scores(tok(question))
    order=np.argsort(scores)[::-1][:pool]
    in_doc=[chunks[i] for i in order if chunks[i]['doc']==document]
    return in_doc[:k] if in_doc else [chunks[i] for i in order[:k]]


## 4. pred_source parser (140/140)


In [ ]:
SOURCE_PATTERNS=[
  (r'dpm\s*2025\s*volume\s*ii|dpm[^.]*vol[^.]*\bii\b|volume\s*2\b','DPM-2025-VOLUME-II.pdf'),
  (r'dpm\s*2025\s*volume\s*i\b|dpm[^.]*vol[^.]*\bi\b|volume\s*1\b','DPM-2025-VOLUME-I.pdf'),
  (r'dfpds|delegation\s+of\s+financial|financial\s+powers','Delegation_of_Financial_Powers_Rules_2024_Booklet.pdf'),
  (r'navy\s*regulations?\s*part\s*iv|\bpart\s*4\b','RegsNavyIV.pdf'),
  (r'navy\s*regulations?\s*part\s*iii|\bpart\s*3\b','RegsNavyIII.pdf'),
  (r'navy\s*regulations?\s*part\s*ii|\bpart\s*2\b','RegsNavyII.pdf'),
  (r'navy\s*regulations?\s*part\s*i\b|\bpart\s*1\b','RegsNavyI.pdf'),
]
def parse_source(q):
    for pat,doc in SOURCE_PATTERNS:
        if re.search(pat,q,re.IGNORECASE): return doc
    return DOCUMENTS[0]


## 5. Load Gemma-4 (E4B, bf16, no-mmap)


In [ ]:
import torch, time
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
COMPUTE=torch.bfloat16   # Gemma overflows fp16; bf16 (fp32-emulated on T4) is correct
kw=dict(device_map='auto', dtype=COMPUTE, disable_mmap=True)
tok_=AutoTokenizer.from_pretrained(GEN_MODEL)
t0=time.time()
try:
    model=AutoModelForCausalLM.from_pretrained(GEN_MODEL, **kw)
except TypeError:
    kw['torch_dtype']=kw.pop('dtype'); kw.pop('disable_mmap',None)
    model=AutoModelForCausalLM.from_pretrained(GEN_MODEL, **kw)
model.eval()
print('loaded in %.0fs, peak VRAM %.1f GB'%(time.time()-t0, torch.cuda.max_memory_allocated()/1e9))


## 6. Grounded generation


In [ ]:
SYSTEM=('You are a precise assistant for Indian defence procurement and policy questions. '
        'Answer ONLY from the provided passages of the cited document. Be concise (1-3 sentences), '
        'factual, and policy-consistent. Do not invent clause numbers. If the passages do not settle '
        'the question, answer with the most relevant governing principle they state.')
def format_evidence(cs, max_chars=2400):
    parts,total=[],0
    for i,c in enumerate(cs,1):
        block='[%d] (%s) %s'%(i, c['section'] or c['doc'], re.sub(r'\s+',' ',c['text']).strip())
        if total+len(block)>max_chars and parts: break
        parts.append(block); total+=len(block)
    return chr(10).join(parts)
def generate(question, document, cs):
    ev=format_evidence(cs)
    user=('Document: '+document+chr(10)+chr(10)+'Passages:'+chr(10)+ev+chr(10)+chr(10)+'Question: '+question+chr(10)+chr(10)+'Grounded answer:')
    msgs=[{'role':'system','content':SYSTEM},{'role':'user','content':user}]
    inp=tok_.apply_chat_template(msgs, add_generation_prompt=True, return_tensors='pt', return_dict=True).to(model.device)
    with torch.inference_mode():
        o=model.generate(**inp, max_new_tokens=160, do_sample=False)
    txt=tok_.decode(o[0][inp['input_ids'].shape[-1]:], skip_special_tokens=True)
    return re.sub(r'\s+',' ',txt).strip()


## 7. Run over 140 questions -> submission.csv


In [ ]:
rows=list(csv.DictReader(open(TEST,encoding='utf-8')))
out=[]
t0=time.time()
for j,r in enumerate(rows):
    q=r['question']; doc=parse_source(q); cs=retrieve_in_doc(q,doc)
    pred=generate(q,doc,cs) or 'The passages do not directly address this question.'
    out.append((r['id'],pred,doc,'section-1'))
    if (j+1)%20==0: print('%d/%d  %.1fs'%(j+1,len(rows),time.time()-t0))
with open('/kaggle/working/submission.csv','w',encoding='utf-8',newline='') as fh:
    w=csv.writer(fh); w.writerow(['id','prediction','pred_source','pred_section']); w.writerows(out)
print('wrote submission.csv', len(out),'rows in %.0fs'%(time.time()-t0)); print(out[0])
